# HKRL Kaggle 离线训练 Notebook

这份 Notebook 与 `one_click_clone_setup.ipynb` 分工明确：前者负责克隆和安装；本 Notebook 只负责检查 rollout、在 Kaggle GPU 上执行一次可审计的 APPO 更新、恢复 checkpoint，并导出下一轮可供 Windows GameWorker 使用的模型。

> 默认 `MODE="inspect"`，只读检查、不复制、不训练。正式训练必须显式改成 `MODE="train"` 并提供真实 rollout NPZ。

## Goal

Kaggle 采用“批次摆渡”而不是实时 SSH 服务：

```text
Windows Hollow Knight + GameWorker
  └─ 使用 policy N 本地推理并采集 rollout_vN.npz
       └─ 上传为 Kaggle Dataset
            └─ 本 Notebook 在 GPU 上更新一次，发布 policy N+1
                 └─ 下载 checkpoint，交回 Windows Worker
```

支持三种模式：

- `inspect`：只检查环境、配置、batch 和 checkpoint；
- `smoke`：生成 4 条合成样本，验证 GPU 更新，不代表模型能力；
- `train`：要求外部提供 NPZ rollout，并进行一次有限 APPO 更新；是否来自真实游戏需由 Windows 采集流程和随附 manifest 证明。

## Setup

### 1. 准备 Kaggle Session

1. 先运行 `one_click_clone_setup.ipynb`，确保 `/kaggle/working/hk_Rl` 已存在且 `hkrl` 已安装；
2. Kaggle Settings 中选择 GPU；本 Notebook 不需要开放监听端口；
3. 正式训练时，把 Windows Worker 生成的 `*.npz` 上传为 Kaggle Dataset；
4. 续训时，同时上传上轮导出的 checkpoint registry 目录（包含 `index.jsonl` 与 `checkpoint_v*.pt`）。

### 2. 修改参数

In [ ]:
MODE = "inspect"  # inspect | smoke | train

REPO_DIR = None  # auto: Kaggle=/kaggle/working/hk_Rl，其他=当前目录
TRAIN_CONFIG = "configs/train/ssh_remote_learner.yaml"
TASK_CONFIGS = ["configs/tasks/gruz_mother.yaml"]

# train 模式必填：只读取此目录第一层的 *.npz。
BATCH_SOURCE_DIR = ""
# train 模式必填：必须是 Windows 采集这些 rollout 时使用的精确 registry。
# inspect/smoke 可留空；smoke 留空时从 policy 0 初始化。
CHECKPOINT_SOURCE_DIR = ""

OUTPUT_ROOT = None  # auto: <repo>/runs
RUN_ID = None  # auto: kaggle-train-UTC时间-随机后缀
MAX_BATCH_FILES = 128
MAX_BATCH_TOTAL_GIB = 8.0
ALLOW_CPU = False

print({
    "mode": MODE,
    "repo_dir": REPO_DIR,
    "train_config": TRAIN_CONFIG,
    "task_configs": TASK_CONFIGS,
    "batch_source_dir": BATCH_SOURCE_DIR or None,
    "checkpoint_source_dir": CHECKPOINT_SOURCE_DIR or None,
    "output_root": OUTPUT_ROOT,
    "allow_cpu": ALLOW_CPU,
})

### Key Assumptions

- PPO/APPO 是近似 on-policy 算法；同一批 rollout 不应反复训练。本 Notebook 每次只执行一次更新。
- 正式 batch 必须与 Windows 采集时加载的精确 checkpoint 配对；仅有相同 policy 版本号还不够。
- `train` 模式不会自行生成合成数据兜底；外部输入缺失、过旧、来自未来 policy 或布局不匹配都会失败。
- NPZ v2 还没有来源证明字段；Notebook 能验证内容与 policy 兼容性，但不能仅凭 NPZ 判断它是否来自真实游戏。
- `/kaggle/input` 视为只读；训练前只把通过校验的 NPZ 和 checkpoint 复制到唯一输出目录。
- checkpoint 仅用 `torch.load(..., weights_only=True)` 读取，并先核对 registry SHA-256。
- 本流程没有真实游戏评测；训练是否有效仍以 Windows 固定种子胜率、受伤量和击杀时间为准。

## Steps

### 3. 环境与配置预检

本格只读取环境和仓库状态。执行模式要求 CUDA，除非显式设置 `ALLOW_CPU=True`。

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import secrets
import shlex
import shutil
import stat
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import torch

mode = MODE.strip().lower()
if mode not in {"inspect", "smoke", "train"}:
    raise ValueError("MODE 必须是 inspect/smoke/train")
if isinstance(MAX_BATCH_FILES, bool) or not isinstance(MAX_BATCH_FILES, int) or MAX_BATCH_FILES <= 0:
    raise ValueError("MAX_BATCH_FILES 必须是正整数")
if not isinstance(MAX_BATCH_TOTAL_GIB, int | float) or isinstance(MAX_BATCH_TOTAL_GIB, bool) or MAX_BATCH_TOTAL_GIB <= 0:
    raise ValueError("MAX_BATCH_TOTAL_GIB 必须是正数")

is_kaggle = bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    or os.environ.get("KAGGLE_URL_BASE")
    or (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir())
)
if REPO_DIR is None:
    repo_candidate = Path("/kaggle/working/hk_Rl") if is_kaggle else Path.cwd()
    if not is_kaggle and not (repo_candidate / "scripts/run_learner.py").is_file():
        parent_candidate = repo_candidate.parent
        if (parent_candidate / "scripts/run_learner.py").is_file():
            repo_candidate = parent_candidate
else:
    repo_candidate = Path(REPO_DIR).expanduser()
repo_dir = repo_candidate.resolve()
if not (repo_dir / "scripts/run_learner.py").is_file():
    raise FileNotFoundError(
        f"HKRL 仓库未准备好: {repo_dir}；请先运行 one_click_clone_setup.ipynb"
    )

python_package = repo_dir / "python"
if str(python_package) not in sys.path:
    sys.path.insert(0, str(python_package))

from hkrl.learner.checkpoint_payload import validate_checkpoint_payload
from hkrl.learner.checkpoint_registry import CheckpointRegistry
from hkrl.models.heads import ACTION_TENSOR_DIM_NO_MACRO
from hkrl.spaces import action_mask_layout, make_observation_space
from hkrl.training.batch_io import BATCH_FORMAT_VERSION, load_rollout_batch, save_rollout_batch
from hkrl.training.rollout_buffer import RolloutBatch
from hkrl.utils.config import load_task_config, load_train_config, validate_task_collection

train_config_path = (repo_dir / TRAIN_CONFIG).resolve()
task_config_paths = [(repo_dir / path).resolve() for path in TASK_CONFIGS]
if not train_config_path.is_file():
    raise FileNotFoundError(train_config_path)
if not task_config_paths or any(not path.is_file() for path in task_config_paths):
    raise FileNotFoundError(f"任务配置不存在: {task_config_paths}")

train_config = load_train_config(train_config_path)
tasks = [load_task_config(path) for path in task_config_paths]
validate_task_collection(tasks, context="Kaggle training tasks")
if train_config.algorithm != "appo":
    raise ValueError("Kaggle batch 训练当前要求 algorithm=appo")

task_layouts = {
    (
        task.observation.max_entities,
        task.observation.tier,
        task.action.enable_macro_actions,
        task.action.n_macro_actions,
    )
    for task in tasks
}
if len(task_layouts) != 1:
    raise ValueError(f"任务模型布局不一致: {sorted(task_layouts)}")
max_entities, observation_tier, enable_macro, n_macro_actions = next(iter(task_layouts))
observation_space = make_observation_space(max_entities, observation_tier)
expected_action_dim = ACTION_TENSOR_DIM_NO_MACRO + int(enable_macro)
expected_mask_dim = len(action_mask_layout(enable_macro, n_macro_actions))
allowed_task_wire_ids = {task.wire_id for task in tasks}

output_root = (
    repo_dir / "runs"
    if OUTPUT_ROOT is None
    else Path(OUTPUT_ROOT).expanduser().resolve()
)
run_id = RUN_ID or (
    "kaggle-train-"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "-"
    + secrets.token_hex(3)
)
if not re.fullmatch(r"[A-Za-z0-9_.-]+", run_id):
    raise ValueError("RUN_ID 只能包含字母、数字、点、下划线和连字符")
run_dir = (output_root / run_id).resolve()
archive_path = (output_root / f"{run_id}-artifacts.zip").resolve()

cuda_available = torch.cuda.is_available()
learner_device = "cuda:0" if cuda_available else "cpu"
if mode != "inspect" and not cuda_available and not ALLOW_CPU:
    raise RuntimeError("CUDA 不可用；Kaggle 请启用 GPU 并重启 Session")

child_env = os.environ.copy()
child_env["PYTHONPATH"] = str(python_package) + os.pathsep + child_env.get("PYTHONPATH", "")

def run_command(command: list[str], *, capture: bool = False) -> subprocess.CompletedProcess[str]:
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=repo_dir,
        env=child_env,
        check=True,
        text=True,
        capture_output=capture,
    )

def json_from_stdout(result: subprocess.CompletedProcess[str]) -> dict[str, Any]:
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, file=sys.stderr, end="")
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError("子进程未输出 JSON")
    payload = json.loads(lines[-1])
    if not isinstance(payload, dict):
        raise RuntimeError("子进程 JSON 不是对象")
    return payload

git_result = subprocess.run(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
)
git_commit = git_result.stdout.strip()
preflight = {
    "mode": mode,
    "is_kaggle": is_kaggle,
    "repo_dir": str(repo_dir),
    "git_commit": git_commit,
    "python": sys.executable,
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": cuda_available,
    "cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "learner_device": learner_device,
    "algorithm": train_config.algorithm,
    "model": train_config.model.name,
    "task_ids": [task.task_id for task in tasks],
    "task_wire_ids": sorted(allowed_task_wire_ids),
    "batch_format_version": BATCH_FORMAT_VERSION,
    "max_staleness": train_config.learner.max_staleness,
    "run_dir": str(run_dir),
}
print(json.dumps(preflight, indent=2, ensure_ascii=False))

### 4. 只读审计 batch 与 checkpoint

本格检查 NPZ 格式、有限值、观察/动作布局、任务 wire id、policy staleness，以及 checkpoint registry 的 SHA-256。不会写入输入目录。

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def audit_checkpoint_registry(path: Path | None) -> dict[str, Any]:
    if path is None:
        return {
            "configured": False,
            "latest_version": 0,
            "latest_policy_version": 0,
            "records": [],
        }
    if not path.is_dir():
        raise NotADirectoryError(path)
    if any(item.is_symlink() for item in path.rglob("*")):
        raise ValueError(f"checkpoint 输入包含符号链接: {path}")
    index_path = path / "index.jsonl"
    if not index_path.is_file():
        raise FileNotFoundError(index_path)
    registry = CheckpointRegistry(str(path))
    latest = registry.latest()
    if latest is None:
        raise ValueError("checkpoint registry 为空")
    records = []
    for version in range(1, latest.version + 1):
        meta = registry.get(version)
        if Path(meta.path).is_absolute():
            raise ValueError("checkpoint index 必须使用相对路径")
        checkpoint_path = registry.resolve_path(meta)
        actual_hash = sha256_file(checkpoint_path)
        if actual_hash != meta.sha256:
            raise ValueError(f"checkpoint {version} SHA-256 不匹配")
        records.append({
            "version": meta.version,
            "policy_version": meta.policy_version,
            "created_step": meta.created_step,
            "file": meta.path,
            "sha256": actual_hash,
            "bytes": checkpoint_path.stat().st_size,
        })
    latest_payload = validate_checkpoint_payload(
        torch.load(registry.resolve_path(latest), map_location="cpu", weights_only=True)
    )
    payload_policy = int(latest_payload.get("policy_version", latest.policy_version))
    if payload_policy != latest.policy_version:
        raise ValueError("latest checkpoint payload 与 index 的 policy_version 不一致")
    return {
        "configured": True,
        "latest_version": latest.version,
        "latest_policy_version": latest.policy_version,
        "optimizer_state_present": "optimizer_state_dict" in latest_payload,
        "records": records,
    }


def audit_rollout(path: Path) -> dict[str, Any]:
    batch = load_rollout_batch(path)
    expected_shapes = {
        "obs_global": observation_space["global"].shape,
        "obs_player": observation_space["player"].shape,
        "obs_entities": observation_space["entities"].shape,
        "entity_mask": observation_space["entity_mask"].shape,
        "actions": (expected_action_dim,),
        "action_masks": (expected_mask_dim,),
    }
    for field, expected in expected_shapes.items():
        actual = tuple(np.asarray(getattr(batch, field)).shape[2:])
        if actual != tuple(expected):
            raise ValueError(f"{path.name}: {field} 尾部形状 {actual} != {tuple(expected)}")
    task_ids = {int(value) for value in np.unique(batch.task_ids)}
    if not task_ids or not task_ids.issubset(allowed_task_wire_ids):
        raise ValueError(f"{path.name}: task_ids {sorted(task_ids)} 不属于配置任务")
    if batch.rnn_states is not None and batch.rnn_states.shape[-1] != train_config.model.rnn_hidden:
        raise ValueError(
            f"{path.name}: rnn hidden {batch.rnn_states.shape[-1]} "
            f"!= {train_config.model.rnn_hidden}"
        )
    return {
        "file": path.name,
        "source_path": str(path),
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
        "samples": int(batch.rewards.size),
        "time_steps": int(batch.rewards.shape[0]),
        "num_envs": int(batch.rewards.shape[1]),
        "policy_version": int(batch.policy_version),
        "task_wire_ids": sorted(task_ids),
        "has_rnn_states": batch.rnn_states is not None,
    }


checkpoint_source_dir = (
    None
    if not CHECKPOINT_SOURCE_DIR.strip()
    else Path(CHECKPOINT_SOURCE_DIR).expanduser().resolve()
)
batch_source_dir = (
    None
    if not BATCH_SOURCE_DIR.strip()
    else Path(BATCH_SOURCE_DIR).expanduser().resolve()
)
if mode == "smoke" and batch_source_dir is not None:
    raise ValueError("smoke 模式不读取 BATCH_SOURCE_DIR；请清空该参数")

checkpoint_audit = audit_checkpoint_registry(checkpoint_source_dir)
if mode == "train" and checkpoint_source_dir is None:
    raise ValueError(
        "train 模式必须提供采集这些 rollout 时使用的 CHECKPOINT_SOURCE_DIR"
    )
source_batch_paths: list[Path] = []
if batch_source_dir is not None:
    if not batch_source_dir.is_dir():
        raise NotADirectoryError(batch_source_dir)
    source_batch_paths = sorted(batch_source_dir.glob("*.npz"))
    if len(source_batch_paths) > MAX_BATCH_FILES:
        raise ValueError(f"batch 文件数 {len(source_batch_paths)} > {MAX_BATCH_FILES}")
    total_bytes = sum(path.stat().st_size for path in source_batch_paths)
    if total_bytes > MAX_BATCH_TOTAL_GIB * 1024**3:
        raise ValueError(f"batch 总大小超过 {MAX_BATCH_TOTAL_GIB} GiB")

batch_audit = [audit_rollout(path) for path in source_batch_paths]
current_policy_version = int(checkpoint_audit["latest_policy_version"])
min_accepted_policy = current_policy_version - train_config.learner.max_staleness
incompatible_batches = [
    record
    for record in batch_audit
    if not min_accepted_policy <= record["policy_version"] <= current_policy_version
]
if incompatible_batches:
    raise ValueError(
        "存在未来或过旧 policy batch: "
        + json.dumps(incompatible_batches, ensure_ascii=False)
    )
if mode == "train" and not batch_audit:
    raise ValueError("train 模式要求 BATCH_SOURCE_DIR 中至少有一个真实 NPZ")
previously_processed_hashes: set[str] = set()
if mode == "train" and output_root.is_dir():
    previous_summaries = sorted(output_root.glob("*/training_summary.json"))
    if len(previous_summaries) > 1000:
        raise RuntimeError("历史训练摘要超过 1000 个，请先归档 OUTPUT_ROOT")
    for summary_path in previous_summaries:
        previous_summary = json.loads(summary_path.read_text(encoding="utf-8"))
        for record in previous_summary.get("input_batches", []):
            digest = record.get("sha256")
            if isinstance(digest, str):
                previously_processed_hashes.add(digest)
    duplicate_hashes = sorted(
        {record["sha256"] for record in batch_audit} & previously_processed_hashes
    )
    if duplicate_hashes:
        raise ValueError(f"拒绝重复训练已处理 batch: {duplicate_hashes}")

input_audit = {
    "mode": mode,
    "checkpoint": checkpoint_audit,
    "batch_count": len(batch_audit),
    "batch_samples": sum(record["samples"] for record in batch_audit),
    "batch_policy_versions": sorted({record["policy_version"] for record in batch_audit}),
    "batches": batch_audit,
    "accepted_policy_range": [min_accepted_policy, current_policy_version],
}
print(json.dumps(input_audit, indent=2, ensure_ascii=False))

### 5. 暂存输入并执行一次 Learner 更新

`inspect` 会跳过本格。`smoke` 生成明确标记的合成 batch；`train` 只复制已经通过审计的真实 batch。每个运行目录唯一，拒绝覆盖。

In [ ]:
def write_synthetic_batch(path: Path, *, policy_version: int) -> Path:
    time_steps, num_envs = 4, 1
    actions = np.zeros((time_steps, num_envs, expected_action_dim), dtype=np.int64)
    actions[:, :, 0:2] = 1
    entity_mask = np.zeros(
        (time_steps, num_envs, *observation_space["entity_mask"].shape),
        dtype=bool,
    )
    entity_mask[:, :, 0] = True
    signal = np.arange(1, time_steps + 1, dtype=np.float32).reshape(time_steps, num_envs)
    batch = RolloutBatch(
        obs_global=np.zeros(
            (time_steps, num_envs, *observation_space["global"].shape), dtype=np.float32
        ),
        obs_player=np.zeros(
            (time_steps, num_envs, *observation_space["player"].shape), dtype=np.float32
        ),
        obs_entities=np.zeros(
            (time_steps, num_envs, *observation_space["entities"].shape), dtype=np.float32
        ),
        entity_mask=entity_mask,
        actions=actions,
        log_probs=np.full((time_steps, num_envs), -1.0, dtype=np.float32),
        values=np.zeros((time_steps, num_envs), dtype=np.float32),
        advantages=signal.copy(),
        returns=signal.copy(),
        rewards=np.ones((time_steps, num_envs), dtype=np.float32),
        dones=np.array([[False], [False], [False], [True]], dtype=bool),
        truncateds=np.zeros((time_steps, num_envs), dtype=bool),
        action_masks=np.ones((time_steps, num_envs, expected_mask_dim), dtype=bool),
        prev_actions=np.zeros((time_steps, num_envs, expected_action_dim), dtype=np.int64),
        prev_rewards=np.zeros((time_steps, num_envs), dtype=np.float32),
        rnn_states=None,
        episode_ids=np.ones((time_steps, num_envs), dtype=np.uint64),
        task_ids=np.full((time_steps, num_envs), tasks[0].wire_id, dtype=np.int64),
        policy_version=policy_version,
    )
    return save_rollout_batch(path, batch)


def make_tree_user_writable(path: Path) -> None:
    for item in [path, *path.rglob("*")]:
        current_mode = item.stat().st_mode
        if item.is_dir():
            item.chmod(current_mode | stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
        elif item.is_file():
            item.chmod(current_mode | stat.S_IRUSR | stat.S_IWUSR)


training_executed = False
learner_summary: dict[str, Any] | None = None
worker_summary: dict[str, Any] | None = None
staged_batch_records: list[dict[str, Any]] = []
training_summary_path: Path | None = None

if mode == "inspect":
    print("inspect 模式：不会创建运行目录、复制输入或启动 Learner。")
else:
    if run_dir.exists() or archive_path.exists():
        raise FileExistsError(f"拒绝覆盖已有运行: {run_dir} / {archive_path}")
    run_dir.mkdir(parents=True, exist_ok=False)
    staged_batch_dir = run_dir / "batches"
    staged_batch_dir.mkdir()
    checkpoint_dir = run_dir / "checkpoints"

    if checkpoint_source_dir is not None:
        shutil.copytree(checkpoint_source_dir, checkpoint_dir)
        make_tree_user_writable(checkpoint_dir)

    if mode == "smoke":
        synthetic_path = write_synthetic_batch(
            staged_batch_dir / "synthetic_smoke_v000000.npz",
            policy_version=current_policy_version,
        )
        staged_batch_records = [audit_rollout(synthetic_path)]
    else:
        for source in source_batch_paths:
            destination = staged_batch_dir / source.name
            if destination.exists():
                raise FileExistsError(destination)
            shutil.copy2(source, destination)
            copied_record = audit_rollout(destination)
            source_record = next(record for record in batch_audit if record["file"] == source.name)
            if copied_record["sha256"] != source_record["sha256"]:
                raise RuntimeError(f"batch 复制后 SHA-256 变化: {source.name}")
            staged_batch_records.append(copied_record)

    learner_result = run_command([
        sys.executable,
        "scripts/run_learner.py",
        "--config",
        str(train_config_path),
        "--tasks",
        *[str(path) for path in task_config_paths],
        "--bind",
        "127.0.0.1:0",
        "--batch-dir",
        str(staged_batch_dir),
        "--checkpoint-dir",
        str(checkpoint_dir),
        "--device",
        learner_device,
        "--publish-every-updates",
        "1",
    ], capture=True)
    learner_summary = json_from_stdout(learner_result)
    expected_batch_count = len(staged_batch_records)
    expected_policy_version = current_policy_version + 1
    expected_checkpoint_version = max(2, int(checkpoint_audit["latest_version"]) + 1)
    if (
        learner_summary.get("submitted_batches") != expected_batch_count
        or learner_summary.get("accepted_batches") != expected_batch_count
        or learner_summary.get("rejected_batches") != 0
        or learner_summary.get("policy_version") != expected_policy_version
        or learner_summary.get("latest_checkpoint") != expected_checkpoint_version
    ):
        raise RuntimeError(f"Learner 更新结果不符合预期: {learner_summary}")

    worker_result = run_command([
        sys.executable,
        "scripts/run_worker.py",
        "--config",
        str(train_config_path),
        "--task",
        str(task_config_paths[0]),
        "--tasks",
        *[str(path) for path in task_config_paths],
        "--registry",
        str(checkpoint_dir),
        "--worker-id",
        "kaggle-offline-validation",
        "--dry-run",
    ], capture=True)
    worker_summary = json_from_stdout(worker_result)
    if worker_summary.get("latest_checkpoint") != learner_summary["latest_checkpoint"]:
        raise RuntimeError("Worker 未加载 Learner 最新 checkpoint")
    training_executed = True

### 6. 验证参数更新并导出

本格核对新 registry、比较更新前后模型参数、记录输入/输出哈希，并生成不含密钥的 JSON 摘要和 ZIP。

In [ ]:
export_summary: dict[str, Any] | None = None
if not training_executed:
    print("没有训练结果需要导出。")
else:
    assert learner_summary is not None
    assert worker_summary is not None
    checkpoint_dir = run_dir / "checkpoints"
    output_checkpoint_audit = audit_checkpoint_registry(checkpoint_dir)
    output_registry = CheckpointRegistry(str(checkpoint_dir))
    latest_meta = output_registry.latest()
    if latest_meta is None or latest_meta.version < 2:
        raise RuntimeError("训练后 checkpoint registry 不完整")
    previous_meta = output_registry.get(latest_meta.version - 1)
    previous_payload = validate_checkpoint_payload(
        torch.load(
            output_registry.resolve_path(previous_meta),
            map_location="cpu",
            weights_only=True,
        )
    )
    latest_payload = validate_checkpoint_payload(
        torch.load(
            output_registry.resolve_path(latest_meta),
            map_location="cpu",
            weights_only=True,
        )
    )
    changed_tensors = 0
    tensor_count = 0
    max_abs_parameter_delta = 0.0
    for name, before in previous_payload["model_state_dict"].items():
        after = latest_payload["model_state_dict"][name]
        if torch.is_tensor(before) and torch.is_tensor(after):
            tensor_count += 1
            delta = float((after - before).abs().max().cpu())
            if delta > 0.0:
                changed_tensors += 1
                max_abs_parameter_delta = max(max_abs_parameter_delta, delta)
    if changed_tensors == 0:
        raise RuntimeError("优化器运行后没有任何模型参数变化")

    portable_batch_records = [
        {key: value for key, value in record.items() if key != "source_path"}
        for record in staged_batch_records
    ]
    export_summary = {
        "ok": True,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "mode": mode,
        "input_origin": "notebook_synthetic" if mode == "smoke" else "external",
        "synthetic_input": True if mode == "smoke" else None,
        "git_commit": git_commit,
        "config": TRAIN_CONFIG,
        "tasks": TASK_CONFIGS,
        "artifact_archive": str(archive_path),
        "device": learner_summary["device"],
        "input_checkpoint": {
            "latest_version": checkpoint_audit["latest_version"],
            "latest_policy_version": checkpoint_audit["latest_policy_version"],
        },
        "input_batches": portable_batch_records,
        "learner": learner_summary,
        "worker_validation": worker_summary,
        "parameter_update": {
            "changed_tensors": changed_tensors,
            "tensor_count": tensor_count,
            "max_abs_delta": max_abs_parameter_delta,
        },
        "output_checkpoint": output_checkpoint_audit,
        "training_metrics": latest_payload.get("metrics", {}),
    }
    training_summary_path = run_dir / "training_summary.json"
    training_summary_path.write_text(
        json.dumps(export_summary, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    archive = shutil.make_archive(
        str(archive_path.with_suffix("")),
        "zip",
        root_dir=run_dir,
    )
    archive_path = Path(archive)
    print(json.dumps({
        "summary": str(training_summary_path),
        "archive": str(archive_path),
        "latest_checkpoint": latest_meta.version,
        "latest_policy_version": latest_meta.policy_version,
        "changed_tensors": changed_tensors,
    }, indent=2, ensure_ascii=False))

## Checks

### 7. 最终状态

成功训练要求 `ok=true`、所有 batch 被接受、策略和 checkpoint 各前进一版、参数发生变化，并且 Worker 能加载新 checkpoint。

In [ ]:
if training_summary_path is None or not training_summary_path.is_file():
    print(json.dumps({
        "mode": mode,
        "preflight_ok": True,
        "input_audit_ok": True,
        "training_executed": False,
        "next": "确认输入后将 MODE 改为 smoke 或 train，再 Run All",
    }, indent=2, ensure_ascii=False))
else:
    final_summary = json.loads(training_summary_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "ok": final_summary["ok"],
        "mode": final_summary["mode"],
        "input_origin": final_summary["input_origin"],
        "synthetic_input": final_summary["synthetic_input"],
        "device": final_summary["device"],
        "accepted_batches": final_summary["learner"]["accepted_batches"],
        "policy_version": final_summary["learner"]["policy_version"],
        "checkpoint_version": final_summary["learner"]["latest_checkpoint"],
        "changed_tensors": final_summary["parameter_update"]["changed_tensors"],
        "summary": str(training_summary_path),
        "archive": str(archive_path),
    }, indent=2, ensure_ascii=False))

## Next Steps

### 首次验证

先设置 `MODE="smoke"` Run All。它只证明 Kaggle GPU、反向传播、优化器和 checkpoint 路径可用。

### 真实训练轮次

1. Windows Worker 用当前最新 checkpoint 采集新的 rollout NPZ，并保留这份 registry；
2. 上传 NPZ，并设置 `MODE="train"`、`BATCH_SOURCE_DIR`；
3. 设置 `CHECKPOINT_SOURCE_DIR` 指向采集这些 NPZ 时使用的精确 registry；
4. Run All 后下载 `<RUN_ID>-artifacts.zip`；
5. 把其中 `checkpoints/` 交给 Windows Worker，再采集下一版 rollout。

> 不要用同一批 NPZ 反复点击训练，也不要把 `smoke` 的合成 checkpoint 当成能力提升。每轮真实提升最终都要回到 Windows 固定种子 evaluator，查看 per-boss 胜率、受伤量、击杀时间和 invalid-action ratio。